# Train the Finder (Stage 1)

Fine-tunes the YOLO26 leaf detector on `data/detect/`, checkpointing every
epoch so an interrupted run can pick up where it left off, then checks the
result against the untouched test set and shows the diagnostic plots right
here -- no separate terminal commands needed.

Needs `data/detect/data.yaml` to exist (run `scripts/split_dataset.py` first
if it doesn't) and `weights/best_m.pt` to be there for a fresh start.

**Round 2 of the augmentation experiment.** Round 1 (`runs/finder/train-4/`)
turned on 4 augmentation settings at once and made things worse -- mAP50-95
dropped from 0.816 to 0.580, boxes got noticeably less tightly fitted even
though precision/recall held up. Too many changes at once to know which one
caused it, so this round isolates just `copy_paste` on its own (see the
config cell for the reasoning). `runs/finder/train/` (the original baseline,
still your best result) is untouched -- this lands in a new folder
automatically, so all three stay around to compare.

In [ ]:
from pathlib import Path

from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_YAML = REPO_ROOT / "data" / "detect" / "data.yaml"
STARTING_WEIGHTS = REPO_ROOT / "weights" / "best_m.pt"   # only used when starting a brand new run
PROJECT_DIR = REPO_ROOT / "runs" / "finder"
RUN_NAME = "train"

EPOCHS = 100
PATIENCE = 40      # stop early if val mAP hasn't improved in this many epochs.
                    # was 20, but that turned out too tight -- with only 112 training images the
                    # val score is noisy epoch to epoch, so a 20-epoch dip isn't necessarily overfitting,
                    # it can just be normal variance. the first real run's val score dipped hard for
                    # a stretch and then recovered and kept improving all the way to epoch 100, which
                    # patience=20 would have (and in one run, actually did) cut off way too early.
IMGSZ = 1280
BATCH = -1          # -1 = let Ultralytics pick the biggest batch that fits your GPU
SAVE_PERIOD = 10   # also keep a numbered snapshot (epoch10.pt, epoch20.pt, ...) every N epochs

# round 1 turned on copy_paste + mixup + degrees + flipud all at once, and it made
# things worse -- mAP50-95 dropped from 0.816 to 0.580 (boxes got sloppier/less tight,
# even though precision/recall held up ok, since those only need IoU>=0.5 to count).
# too many changes at once to know which one did it, so this round isolates just
# copy_paste on its own -- it's the one that actually targets the dense-cluster
# weakness the test set showed. the other three are back off.
COPY_PASTE = 0.3   # synthesizes overlapping-leaf examples by pasting instances onto other images --
                    # directly targets the dense-cluster weakness.
                    # (0.3 matches what the original leaf-localisation model itself was trained with)
MIXUP = 0.0         # was 0.15 in round 1 -- off now, isolating copy_paste alone
DEGREES = 0.0       # was 45.0 in round 1 -- off now. worth revisiting later with a milder angle
                    # (e.g. 15) once we know whether copy_paste alone even helps
FLIPUD = 0.0        # was 0.5 in round 1 -- off now, same reasoning

### about resuming

Ultralytics writes `weights/last.pt` after *every single epoch*, with the
full optimizer state included, not just the model weights -- that's on by
default, nothing extra to turn on. So if this kernel dies, your machine
reboots, whatever -- nothing's actually lost. Just re-run the cell below and
it'll notice the unfinished run and continue from wherever it stopped instead
of starting over from epoch 0.

One thing worth knowing: resuming continues with whatever settings that run
was *originally* started with. If you tweak `EPOCHS` / `PATIENCE` above and
then resume a run that's already in progress, those edits get ignored until
the run either finishes or you start a fresh one.

In [ ]:
last_checkpoint = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"


def start_fresh():
    model = YOLO(str(STARTING_WEIGHTS))
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        save_period=SAVE_PERIOD,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        copy_paste=COPY_PASTE,
        mixup=MIXUP,
        degrees=DEGREES,
        flipud=FLIPUD,
    )
    return model


if last_checkpoint.exists():
    try:
        print(f"found a checkpoint at {last_checkpoint} -- trying to resume it")
        model = YOLO(str(last_checkpoint))
        # important: pass the checkpoint PATH here, not just True -- passing bare
        # True makes Ultralytics go hunting for "the most recently modified run"
        # anywhere under runs/, which is not necessarily this one
        model.train(resume=str(last_checkpoint))
    except AssertionError as e:
        print(f"nothing to resume ({e})")
        print("starting a fresh run instead -- it'll land in a new numbered folder, nothing gets overwritten")
        model = start_fresh()
else:
    print("no existing checkpoint found, starting fresh")
    model = start_fresh()

save_dir = model.trainer.save_dir
print(f"\nthis run's files are in: {save_dir}")

### now check it against the test set

The number Ultralytics just printed above is the **val** score -- that's the
same 14 photos it checks after every epoch to decide which checkpoint
becomes `best.pt`, so it's a little bit optimistic (the checkpoint was
literally selected because it did well on those photos). The **test** split
(14 different photos) was never touched during training at all. This is the
honest, unbiased number.

In [ ]:
# project/name here matter -- without them Ultralytics saves to a default
# location based on wherever this notebook's working directory happens to be
# (which is notebooks/, not the repo root), scattering results away from
# everything else about this run. Nesting it under save_dir keeps it together.
test_metrics = model.val(
    data=str(DATA_YAML), split="test", imgsz=IMGSZ,
    project=str(save_dir), name="test",
)
print(
    f"\ntest set -- precision: {test_metrics.box.mp:.3f}  recall: {test_metrics.box.mr:.3f}  "
    f"mAP50: {test_metrics.box.map50:.3f}  mAP50-95: {test_metrics.box.map:.3f}"
)
test_dir = save_dir / "test"
print(f"test set plots saved to: {test_dir}")

### what actually went right or wrong

These get written automatically during training/validation -- pulling them
up here instead of digging through the run folder by hand. First the training
run's own curves, then the held-out test set's predictions vs. ground truth
side by side (the more honest one to actually judge the model by).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def show(path, title):
    if not path.exists():
        print(f"(missing: {path})")
        return
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(path))
    plt.title(title)
    plt.axis("off")
    plt.show()


show(save_dir / "results.png", "training curves (all epochs)")
show(save_dir / "confusion_matrix.png", "confusion matrix (val split)")
show(test_dir / "val_batch0_labels.jpg", "test set -- ground truth")
show(test_dir / "val_batch0_pred.jpg", "test set -- what the model predicted")

### next

**Compare against the first run before trusting this one.** More
augmentation isn't automatically better -- it can just as easily make things
worse on a dataset this small. Put the two side by side:

- baseline's test mAP50 / mAP50-95 (0.854 / 0.754): `runs/finder/train/test/`
- round 1's, for reference -- came out worse, mAP50-95 dropped to ~0.58: `runs/finder/train-4/`
- this run's: printed above, saved next to `save_dir` (the folder printed by the cell above)

If this run's numbers are actually higher, this is your new Stage 1
checkpoint. If not, the augmentation changes weren't the right lever here --
`runs/finder/train/weights/best.pt` (the original) is still sitting there
untouched, so nothing is lost either way.

If it's still not enough, or the plots show the same dense-cluster weakness,
that's the README's step 9: go get more raw photos that specifically cover
whatever's failing, run them through `scripts/auto_label.py` + CVAT +
`scripts/split_dataset.py`, and re-run this notebook. Since it auto-detects
finished runs, it's safe to just re-run rather than needing to clean
anything up first.